In [2]:
import joblib
import pandas as pd
import numpy as np

# Load model dan data
model = joblib.load('../models/random_forest_final.pkl')
df    = pd.read_csv('../data/processed/ravenstack_features_for_modeling.csv')
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

X = df.drop(columns=['target'])
y = df['target']

# Cek probabilitas semua pelanggan
proba_all = model.predict_proba(X)[:, 1]
print('Distribusi probabilitas:')
print(pd.Series(proba_all).describe())
print()
print('Berapa yang > 0.33:', (proba_all >= 0.33).sum())
print('Berapa yang > 0.50:', (proba_all >= 0.50).sum())
print('Berapa yang > 0.70:', (proba_all >= 0.70).sum())
print()

# Cek profil Enterprise Matang
base = X.mean().to_dict()
base.update({
    'tenure_days': 600, 'total_mrr': 1200,
    'days_since_last_usage': 1, 'unique_features_used': 32,
    'avg_satisfaction_score': 4.9, 'total_tickets': 1,
    'escalation_rate': 0.0, 'sub_churn_ratio': 0.0,
    'net_plan_movement': 4, 'seats': 50,
})
X_test = pd.DataFrame([base]).reindex(columns=X.columns, fill_value=0)
prob = model.predict_proba(X_test)[0, 1]
print(f'Enterprise Matang probabilitas: {prob:.4f}')

Distribusi probabilitas:
count    500.000000
mean       0.329997
std        0.167417
min        0.106734
25%        0.206763
50%        0.274149
75%        0.426732
max        0.794514
dtype: float64

Berapa yang > 0.33: 184
Berapa yang > 0.50: 92
Berapa yang > 0.70: 12

Enterprise Matang probabilitas: 0.7172


In [3]:
# Cek fitur mana yang paling mendorong prediksi tinggi
import pandas as pd
import numpy as np

base = X.mean().to_dict()
base.update({
    'tenure_days': 600, 'total_mrr': 1200,
    'days_since_last_usage': 1, 'unique_features_used': 32,
    'avg_satisfaction_score': 4.9, 'total_tickets': 1,
    'escalation_rate': 0.0, 'sub_churn_ratio': 0.0,
    'net_plan_movement': 4, 'seats': 50,
    'feature_breadth_ratio': 32/40,
    'usage_density': 12.0,
    'pct_urgent': 0.0,
    'ever_downgraded': 0,
    'has_low_satisfaction': 0,
    'num_upgrades': 4,
    'num_downgrades': 0,
})

X_test = pd.DataFrame([base]).reindex(columns=X.columns, fill_value=0)

# Cek nilai fitur penting
important_feats = ['error_rate', 'avg_error_count', 'sub_churn_ratio',
                   'ever_downgraded', 'tenure_days', 'avg_satisfaction_score',
                   'days_since_last_usage', 'feature_breadth_ratio',
                   'avg_first_response_mins', 'pct_annual_billing']

print('Nilai fitur penting di Enterprise Matang:')
for f in important_feats:
    if f in X_test.columns:
        mean_val = X[f].mean()
        test_val = X_test[f].values[0]
        print(f'  {f:35}: val={test_val:.4f} | mean={mean_val:.4f}')

Nilai fitur penting di Enterprise Matang:
  error_rate                         : val=0.5640 | mean=0.5640
  avg_error_count                    : val=0.5640 | mean=0.5640
  sub_churn_ratio                    : val=0.0000 | mean=0.0965
  ever_downgraded                    : val=0.0000 | mean=0.3540
  tenure_days                        : val=600.0000 | mean=339.7260
  avg_satisfaction_score             : val=4.9000 | mean=3.6946
  days_since_last_usage              : val=1.0000 | mean=17.3120
  feature_breadth_ratio              : val=0.8000 | mean=0.6904
  avg_first_response_mins            : val=87.1431 | mean=87.1431
  pct_annual_billing                 : val=0.4913 | mean=0.4913


In [1]:
import joblib, pandas as pd, numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split

model = joblib.load('../models/random_forest_final.pkl')
df = pd.read_csv('../data/processed/ravenstack_features_for_modeling.csv')
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

proba = model.predict_proba(X_test)[:, 1]

print(f'{"Threshold":>10} {"Recall":>8} {"Precision":>10} {"F1":>8} {"Pred Churn":>12}')
print('-' * 55)
for t in [0.30, 0.33, 0.35, 0.38, 0.40, 0.42, 0.45, 0.50]:
    pred = (proba >= t).astype(int)
    rec  = recall_score(y_test, pred, zero_division=0)
    prec = precision_score(y_test, pred, zero_division=0)
    f1   = f1_score(y_test, pred, zero_division=0)
    print(f'{t:>10.2f} {rec:>8.3f} {prec:>10.3f} {f1:>8.3f} {pred.sum():>12}')

 Threshold   Recall  Precision       F1   Pred Churn
-------------------------------------------------------
      0.30    0.864      0.284    0.427           67
      0.33    0.864      0.339    0.487           56
      0.35    0.727      0.314    0.438           51
      0.38    0.636      0.389    0.483           36
      0.40    0.591      0.394    0.473           33
      0.42    0.500      0.355    0.415           31
      0.45    0.227      0.278    0.250           18
      0.50    0.136      0.375    0.200            8


In [2]:
import joblib, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

model = joblib.load('../models/random_forest_final.pkl')
df = pd.read_csv('../data/processed/ravenstack_features_for_modeling.csv')
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

proba_train = model.predict_proba(X_train)[:, 1]
proba_test  = model.predict_proba(X_test)[:, 1]
proba_all   = model.predict_proba(X)[:, 1]

print('=== Distribusi Probabilitas ===')
print(f'{"":15} {"Train (400)":>12} {"Test (100)":>12} {"All (500)":>12}')
print('-' * 55)
print(f'{"Mean":15} {proba_train.mean():>12.4f} {proba_test.mean():>12.4f} {proba_all.mean():>12.4f}')
print(f'{"Std":15} {proba_train.std():>12.4f} {proba_test.std():>12.4f} {proba_all.std():>12.4f}')
print(f'{"Min":15} {proba_train.min():>12.4f} {proba_test.min():>12.4f} {proba_all.min():>12.4f}')
print(f'{"Max":15} {proba_train.max():>12.4f} {proba_test.max():>12.4f} {proba_all.max():>12.4f}')
print()
print('=== Churn Rate Aktual ===')
print(f'Train churn rate: {y_train.mean():.4f} ({y_train.sum()} dari {len(y_train)})')
print(f'Test churn rate : {y_test.mean():.4f} ({y_test.sum()} dari {len(y_test)})')
print(f'All churn rate  : {y.mean():.4f} ({y.sum()} dari {len(y)})')
print()
print('=== Probabilitas > threshold per set ===')
for t in [0.33, 0.40, 0.50]:
    tr = (proba_train >= t).sum()
    te = (proba_test  >= t).sum()
    al = (proba_all   >= t).sum()
    print(f'threshold {t}: Train={tr} ({tr/len(proba_train)*100:.1f}%) | Test={te} ({te/len(proba_test)*100:.1f}%) | All={al} ({al/len(proba_all)*100:.1f}%)')

=== Distribusi Probabilitas ===
                 Train (400)   Test (100)    All (500)
-------------------------------------------------------
Mean                  0.3238       0.3547       0.3300
Std                   0.1802       0.0961       0.1672
Min                   0.1067       0.1767       0.1067
Max                   0.7945       0.5437       0.7945

=== Churn Rate Aktual ===
Train churn rate: 0.2200 (88 dari 400)
Test churn rate : 0.2200 (22 dari 100)
All churn rate  : 0.2200 (110 dari 500)

=== Probabilitas > threshold per set ===
threshold 0.33: Train=128 (32.0%) | Test=56 (56.0%) | All=184 (36.8%)
threshold 0.4: Train=99 (24.8%) | Test=33 (33.0%) | All=132 (26.4%)
threshold 0.5: Train=84 (21.0%) | Test=8 (8.0%) | All=92 (18.4%)
